# RTE results: tests and figures of the paper

Reads the test predictions in `results/rte/` (written by `scripts/run.py`) and produces:
1. the Diebold–Mariano p-values for all pairs of models;
2. the heat maps of the Diebold–Mariano statistics;
3. forecasts for example week;
4. the RMSE by month.

TiDE: the tests use the average of the seeds' **errors**; the forecasts plotted are those of the **best seed**.
The test metrics themselves (RMSE, MAE, MAPE) are in `results/rte/<model>/metrics.json`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import yaml

from sigadapt.metrics import diebold_mariano

In [ ]:
# test predictions as written by scripts/run.py: results/rte/<model>/...
parent_dir = Path.cwd().parent
results_dir = parent_dir / "results" / "rte"
analysis_dir = results_dir / "analysis"            # tables produced by this notebook
analysis_dir.mkdir(parents=True, exist_ok=True)
data_file = parent_dir / "data" / "rte_2018_2026.csv"
fig_dir = parent_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

files = {"SigAdapt": "sigadapt/predictions_test.csv", "GAM": "gam/predictions_test_static.csv",
         "GAMAdapt": "gam/predictions_test.csv", "Chronos-2": "chronos2/predictions_test.csv"}
seeds = yaml.safe_load(open(parent_dir / "configs" / "rte" / "tide.yaml"))["seeds"]   # TiDE seeds
models = ["SigAdapt", "TiDE", "GAMAdapt", "GAM", "Chronos-2"]
dm_lag = 3

colours = {"SigAdapt": "#c0392b", "TiDE": "#2c6fbb", "GAMAdapt": "#3a9d5d",
           "GAM": "#9aa5b1", "Chronos-2": "#e08e2b", "Observed": "black"}
plt.rcParams.update({"font.family": "serif", "font.size": 9, "pdf.fonttype": 42,
                     "axes.spines.top": False, "axes.spines.right": False})

## Data

In [ ]:
def load(name):
    return pd.read_csv(results_dir / name, parse_dates=["DateTime"]).set_index("DateTime").sort_index()

pred = {m: load(f)["y_pred"] for m, f in files.items()}
y = load(files["SigAdapt"])["y_true"]
tide = pd.DataFrame({s: load(f"tide/predictions_test_seed{s}.csv")["y_pred"] for s in seeds})

for name, f in list(files.items()) + [(f"TiDE seed {s}", f"tide/predictions_test_seed{s}.csv") for s in seeds]:
    p = load(f)
    assert p.index.equals(y.index) and np.allclose(p["y_true"], y), name

err = {m: y - p for m, p in pred.items()}
tide_err = tide.rsub(y, axis=0)

sq = {m: e ** 2 for m, e in err.items()} | {"TiDE": (tide_err ** 2).mean(axis=1)}
ab = {m: e.abs() for m, e in err.items()} | {"TiDE": tide_err.abs().mean(axis=1)}

tide_rmse = np.sqrt((tide_err ** 2).mean())
best_seed = tide_rmse.idxmin()
pred["TiDE"] = tide[best_seed]

temp = pd.read_csv(data_file, usecols=["DateTime", "Temperature"], parse_dates=["DateTime"]) \
         .set_index("DateTime")["Temperature"]

print(f"{len(y)} half-hours, {y.index[0]} to {y.index[-1]}; best TiDE seed for plots: {best_seed}")

## 1. Diebold–Mariano p-values

Loss differential averaged per day, Newey–West variance with a lag of `dm_lag` days, **one-sided** p-values
(`diebold_mariano` in `src/sigadapt/metrics.py`).
Row vs column: a negative statistic means the row model is more accurate; the p-value is that of
H0 "the row model is not more accurate than the column model", so a small p-value means the row model is
significantly more accurate.

In [ ]:
dm_stat, dm_p = {}, {}
for loss_name, losses in [("squared", sq), ("absolute", ab)]:
    s_tab = pd.DataFrame(np.nan, index=models, columns=models)
    p_tab = s_tab.copy()
    for a in models:
        for b in models:
            if a != b:
                s_tab.loc[a, b], p_tab.loc[a, b] = diebold_mariano(losses[a], losses[b], lag=dm_lag, one_sided=True)
    dm_stat[loss_name], dm_p[loss_name] = s_tab, p_tab
    s_tab.to_csv(analysis_dir / f"dm_statistic_{loss_name}.csv")
    p_tab.to_csv(analysis_dir / f"dm_pvalue_{loss_name}.csv")
    print(f"\n=== {loss_name} loss: p-values ===")
    print(p_tab.map(lambda v: "" if pd.isna(v) else f"{v:.2g}").to_string())

## 2. Heat maps of the Diebold–Mariano statistics

Row vs column; blue: the row model is more accurate, red: less accurate. Stars (one-sided p-values): the row model is significantly more accurate than the column model, * p < 0.05, ** p < 0.01, *** p < 0.001.

In [ ]:
def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.1), gridspec_kw={"wspace": 0.08})
lim = max(np.nanmax(np.abs(dm_stat[k].to_numpy(dtype=float))) for k in dm_stat)
for ax, loss_name in zip(axes, ["squared", "absolute"]):
    s_tab, p_tab = dm_stat[loss_name], dm_p[loss_name]
    im = ax.imshow(s_tab.to_numpy(dtype=float), cmap="RdBu_r", vmin=-lim, vmax=lim)
    for i, a in enumerate(models):
        for j, b in enumerate(models):
            if a != b:
                v = s_tab.loc[a, b]
                ax.text(j, i, f"{v:.1f}{stars(p_tab.loc[a, b])}", ha="center", va="center", fontsize=6.5,
                        color="white" if abs(v) > 0.6 * lim else "black")
    ax.set_xticks(range(len(models)), models, rotation=45, ha="right")
    ax.set_yticks(range(len(models)), models if ax is axes[0] else [])
    ax.set_title(f"{loss_name.capitalize()} loss", fontsize=9)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(True)
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="DM statistic (row vs column)")
fig.savefig(fig_dir / "dm_heatmap.pdf", bbox_inches="tight")
plt.show()

## 3. Example week

By default, the test week (outside 20 Dec–7 Jan) with the largest range of daily mean temperature.
Set `week_start` to a date string to choose another week.

In [ ]:
week_start = None

daily_temp = temp.reindex(y.index).resample("D").mean()
if week_start is None:
    ranges = daily_temp.rolling(7).max() - daily_temp.rolling(7).min()
    ends = ranges.index
    xmas = ((ends.month == 12) & (ends.day >= 14)) | ((ends.month == 1) & (ends.day <= 13))
    week_end = ranges[~xmas].idxmax()
    week_start = week_end - pd.Timedelta(days=6)
start = pd.Timestamp(week_start)
end = start + pd.Timedelta(days=7) - pd.Timedelta(minutes=30)

fig, (ax, ax_t) = plt.subplots(2, 1, figsize=(7.0, 3.6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.plot(y.loc[start:end] / 1000, color=colours["Observed"], lw=1.6, label="Observed")
week_rmse = {}
for m in ["SigAdapt", "TiDE", "GAMAdapt", "Chronos-2"]:
    week_rmse[m] = np.sqrt(((y - pred[m]).loc[start:end] ** 2).mean())
    ax.plot(pred[m].loc[start:end] / 1000, color=colours[m], lw=1, label=f"{m} ({week_rmse[m]:,.0f})")
ax.set_ylabel("Load (GW)")
ax.legend(frameon=False, fontsize=7.5, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
ax_t.plot(temp.loc[start:end], color="#555555", lw=1)
ax_t.set_ylabel("Temp. (°C)")
ax_t.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
fig.tight_layout()
fig.savefig(fig_dir / "example_week.pdf", bbox_inches="tight")
plt.show()

print(f"Week {start.date()} to {end.date()}; RMSE (MW), TiDE = seed {best_seed}:")
print({m: round(float(v)) for m, v in week_rmse.items()})

## 4. RMSE by month (2024–2026 pooled)

In [ ]:
month = y.index.month
monthly = pd.DataFrame({m: np.sqrt(sq[m].groupby(month).mean()) for m in models})
monthly.index.name = "month"
print(monthly.round(0).to_string())
monthly.to_csv(analysis_dir / "rmse_by_month.csv")

fig, ax = plt.subplots(figsize=(7.0, 2.6))
for m in models:
    ax.plot(monthly.index, monthly[m], "o-", color=colours[m], label=m, ms=3.5,
            lw=1.6 if m == "SigAdapt" else 1)
ax.set_xticks(range(1, 13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_ylabel("RMSE (MW)")
ax.legend(frameon=False, fontsize=7.5, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.15))
fig.tight_layout()
fig.savefig(fig_dir / "rmse_by_month.pdf", bbox_inches="tight")
plt.show()